### 1. Initialization and read


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length, date_sub, lead


In [0]:
df = spark.table("catalogue_project1.bronze.erp_loc_a101")

In [0]:
%sql

select distinct cntry from catalogue_project1.bronze.erp_loc_a101

### 2. Transformations
2.1 Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


2.2 Normalize customer ID

In [0]:
df = df.withColumn(
    "cid", 
    F.regexp_replace(col("cid"), "-", "")
)

2.3 Standarize countries names

In [0]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

2.4 Rename columns

In [0]:
RENAME_CONFIG = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_CONFIG.items():
    df = df.withColumnRenamed(old_name, new_name)

* Intermediate step: check the changes applied correctly.

In [0]:
df.limit(10).display()

### 3. Write into the Silver delta table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("catalogue_project1.silver.erp_loc_a101")

* Check that everything went well

In [0]:
%sql
SELECT * FROM catalogue_project1.silver.erp_loc_a101 LIMIT 10
